# Scratch vs Keras

## Import Libraries

In [10]:
import os
import sys

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

from sklearn.metrics import f1_score, classification_report

from src.cnn.layers.conv2d import Conv2DLayer
from src.cnn.layers.flatten import FlattenLayer
# from src.cnn.layers.locally_connected import LocallyConnected2DLayer
from src.cnn.layers.pooling import MaxPooling2DLayer, AveragePooling2DLayer
from src.cnn.utils import load_image, batch_load_images
from src.shared.layer import Dense
from src.cnn.model import CNNModel


SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))
tf.test.is_built_with_cuda()

TF version: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


True

## Config

In [11]:
IMG_SIZE    = (150, 150)
BATCH_SIZE  = 32
EPOCHS      = 10
NUM_CLASSES = 6
CLASS_NAMES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
LR          = 1e-4
VAL_SPLIT   = 0.2
AUTOTUNE    = tf.data.AUTOTUNE

TRAIN_DIR = '../../data/intel/seg_train'
TEST_DIR  = '../../data/intel/seg_test'

## Load Model Keras

In [12]:
keras_model = keras.models.load_model('Layer-2-32-64-5x5-maxpool.h5')
keras_model.summary()


I0000 00:00:1778753874.872324    3184 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "Layer-2-32-64-5x5-maxpool"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2s_1 (Conv2D)               │ (None, 146, 146, 32)   │         2,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_1 (MaxPooling2D)        │ (None, 73, 73, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2s_2 (Conv2D)               │ (None, 69, 69, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_2 (MaxPooling2D)        │ (None, 34, 34, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 73984)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │     9,470,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,524,552 (36.33 MB)

 Trainable params: 9,524,550 (36.33 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)


>>> Daftar Layer: 
Conv2D                    - conv2s_1
MaxPooling2D              - maxpool_1
Conv2D                    - conv2s_2
MaxPooling2D              - maxpool_2
Flatten                   - flatten
Dense                     - dense_1
Dense                     - output


## Load Seluruh Test Set

In [13]:
normalization = keras.layers.Rescaling(1.0 / 255.0)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False,
)

test_ds_norm = (
    test_ds
    .map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
    .prefetch(buffer_size=AUTOTUNE)
)

print('>>> Mengumpulkan seluruh test set ke NumPy...')
X_list, y_list = [], []
for X_batch, y_batch in test_ds_norm:
    X_list.append(X_batch.numpy())
    y_list.append(y_batch.numpy())

X_test = np.concatenate(X_list, axis=0)
y_true = np.concatenate(y_list, axis=0).astype(int)
print('>>> Selesai mengumpulkan test set.')
print(f'X_test : {X_test.shape}, dtype={X_test.dtype}')
print(f'y_true : {y_true.shape}')
print(f'Distribusi kelas: {np.bincount(y_true).tolist()}')
print(f'Kelas: {test_ds.class_names}')

assert X_test.min() >= 0.0 and X_test.max() <= 1.0
print('\nSanity check OK — nilai pixel dalam [0, 1]')

Found 3000 files belonging to 6 classes.
>>> Mengumpulkan seluruh test set ke NumPy...
>>> Selesai mengumpulkan test set.
X_test : (3000, 150, 150, 3), dtype=float32
y_true : (3000,)
Distribusi kelas: [437, 474, 553, 525, 510, 501]
Kelas: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

Sanity check OK — nilai pixel dalam [0, 1]


## Forward Pass Keras

In [14]:
t_keras_start = time.perf_counter()
y_prob_keras  = keras_model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
t_total_keras = time.perf_counter() - t_keras_start

y_pred_keras = np.argmax(y_prob_keras, axis=1)
f1_keras     = f1_score(y_true, y_pred_keras, average='macro')

print(f'\n{"="*50}')
print(f'KERAS — Macro F1 : {f1_keras:.4f}')
print(f'Waktu inferensi : {t_total_keras:.2f} detik')
print(f'{"="*50}')
print(classification_report(y_true, y_pred_keras, target_names=CLASS_NAMES))

W0000 00:00:1778753923.163911    3184 cpu_allocator_impl.cc:82] Allocation of 810000000 exceeds 10% of free system memory.
W0000 00:00:1778753945.370375    3184 cpu_allocator_impl.cc:82] Allocation of 810000000 exceeds 10% of free system memory.
I0000 00:00:1778753950.641356    7161 service.cc:153] XLA service 0x7de91402e610 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778753950.641432    7161 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 12.5.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.22.0)
I0000 00:00:1778753951.194186    7161 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778753951.562089    7161 cuda_dnn.cc:461] Loaded cuDNN version 92200


 2/94 ━━━━━━━━━━━━━━━━━━━━ 8s 89ms/step  

I0000 00:00:1778753960.726189    7161 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


94/94 ━━━━━━━━━━━━━━━━━━━━ 15s 47ms/step

KERAS — Macro F1 : 0.7846
Waktu inferensi : 52.92 detik
              precision    recall  f1-score   support

   buildings       0.78      0.70      0.74       437
      forest       0.93      0.89      0.91       474
     glacier       0.77      0.73      0.75       553
    mountain       0.76      0.72      0.74       525
         sea       0.72      0.82      0.77       510
      street       0.76      0.84      0.80       501

    accuracy                           0.78      3000
   macro avg       0.79      0.78      0.78      3000
weighted avg       0.79      0.78      0.78      3000



## Pipeline Scratch

In [ ]:
scratch_layers = [
    Conv2DLayer(keras_model.get_layer('conv2s_1')),
    MaxPooling2DLayer(keras_model.get_layer('maxpool_1')),
    Conv2DLayer(keras_model.get_layer('conv2s_2')),
    MaxPooling2DLayer(keras_model.get_layer('maxpool_2')),
    FlattenLayer(),
]


test_probe = X_test[0] 
for layer in scratch_layers:
    test_probe = layer.forward(test_probe)

flattened_size = test_probe.shape[0] 

dense_1 = Dense(flattened_size, keras_model.get_layer('dense_1').units)
dense_2 = Dense(keras_model.get_layer('dense_1').units, keras_model.get_layer('output').units)

# Transfer weights from Keras model to scratch layers
keras_dense_1_weights, keras_dense_1_bias = keras_model.get_layer('dense_1').get_weights()
keras_dense_2_weights, keras_dense_2_bias = keras_model.get_layer('output').get_weights()

# Debug: Check shapes
print(f'Dense 1 - Expected shape: {(flattened_size, keras_model.get_layer("dense_1").units)}, Got: {keras_dense_1_weights.shape}')
print(f'Dense 2 - Expected shape: {(keras_model.get_layer("dense_1").units, keras_model.get_layer("output").units)}, Got: {keras_dense_2_weights.shape}')

# Transpose if necessary (Keras might store them as (out_features, in_features))
if keras_dense_1_weights.shape != (flattened_size, keras_model.get_layer('dense_1').units):
    keras_dense_1_weights = keras_dense_1_weights.T
if keras_dense_2_weights.shape != (keras_model.get_layer('dense_1').units, keras_model.get_layer('output').units):
    keras_dense_2_weights = keras_dense_2_weights.T

dense_1.set_weights(keras_dense_1_weights, keras_dense_1_bias)
dense_2.set_weights(keras_dense_2_weights, keras_dense_2_bias)

scratch_layers.extend([dense_1, dense_2])

model_scratch = CNNModel.from_layers(scratch_layers)

print('>>> Pipeline from scratch:')
for i, layer in enumerate(model_scratch.layers):
    print(f'  [{i}] {type(layer).__name__}')

# Sanity check satu gambar sebelum jalankan full test set
probe_out = model_scratch.forward(X_test[0])
assert probe_out.shape == (6,), f'Output shape salah: {probe_out.shape}'
assert abs(probe_out.sum() - 1.0) < 1e-5, f'Tidak sum to 1: {probe_out.sum()}'
assert not np.any(np.isnan(probe_out)), 'NaN terdeteksi di output!'
print('\nSanity check pipeline OK')

Dense 1 - Expected shape: (34, 128), Got: (73984, 128)
Dense 2 - Expected shape: (128, 6), Got: (128, 6)


AssertionError: 

## Forward Pass Scratch

In [ ]:
y_pred_scratch = np.zeros(len(X_test), dtype=int)
t_start = time.perf_counter()

for i, img in enumerate(X_test):
    y_pred_scratch[i] = np.argmax(model_scratch.forward(img))
    if (i + 1) % 500 == 0:
        print(f'  {i+1}/{len(X_test)}')

t_scratch = time.perf_counter() - t_start
f1_scratch = f1_score(y_true, y_pred_scratch, average='macro')

print(f'\nFrom Scratch Macro F1 : {f1_scratch:.4f}')
print(f'Waktu                 : {t_scratch:.2f} detik')
print(classification_report(y_true, y_pred_scratch,
      target_names=test_ds.class_names))

## Perbandingan Hasil

In [ ]:
n_beda = int(np.sum(y_pred_keras != y_pred_scratch))

print(f'Keras F1       : {f1_keras:.4f}')
print(f'From Scratch F1: {f1_scratch:.4f}')
print(f'Selisih F1     : {abs(f1_keras - f1_scratch):.6f}')
print(f'Prediksi beda  : {n_beda} / {len(X_test)} gambar ({n_beda/len(X_test)*100:.2f}%)')

# LocallyConneceted2D

In [ ]:
# IMG_SIZE_LC = (64, 64)

# keras_lc = keras.models.load_model('locally_connected.h5')
# keras_lc.summary()

# print("\n>>> Daftar Layer: ")
# for l in keras_model.layers:
#     print(f"{type(l).__name__:<25} - {l.name}")

In [ ]:
# test_ds_lc = tf.keras.utils.image_dataset_from_directory(
#     TEST_DIR,
#     image_size=IMG_SIZE_LC,
#     batch_size=BATCH_SIZE,
#     label_mode='int',
#     shuffle=False,
# )

# test_ds_lc_norm = (
#     test_ds_lc
#     .map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
#     .prefetch(buffer_size=AUTOTUNE)
# )

# X_lc_list, y_lc_list = [], []
# for X_b, y_b in test_ds_lc_norm:
#     X_lc_list.append(X_b.numpy())
#     y_lc_list.append(y_b.numpy())

# X_test_lc = np.concatenate(X_lc_list, axis=0) 
# y_true_lc = np.concatenate(y_lc_list, axis=0).astype(int)

# print(f'X_test_lc : {X_test_lc.shape}')

In [ ]:
# t_lc_keras_start  = time.perf_counter()
# y_prob_lc_keras   = keras_lc.predict(X_test_lc, batch_size=BATCH_SIZE, verbose=1)
# t_lc_keras        = time.perf_counter() - t_lc_keras_start

# y_pred_lc_keras = np.argmax(y_prob_lc_keras, axis=1)
# f1_lc_keras     = f1_score(y_true_lc, y_pred_lc_keras, average='macro')

# print(f'\nKeras LC2D — Macro F1 : {f1_lc_keras:.4f}')
# print(classification_report(y_true_lc, y_pred_lc_keras, target_names=CLASS_NAMES))